In [ ]:
import pandas as pd
from datetime import datetime
from rich import print
from rich.pretty import Pretty

pd.set_option('display.max_columns', None)

In [ ]:
df_work_raw = catalog.load('raw/openalex/work_dev#parquet')

**Estructura de Authorships**

In [ ]:
from rich import print, pretty

first_authorship = df_work_raw['authorships'].iloc[0][0]
print(pretty.Pretty(first_authorship, expand_all=False))

# Transformaciones

In [ ]:
# Seleccionar las columnas necesarias y convertir los tipos de datos
df_work2authorships = df_work_raw[['id', 'authorships']].convert_dtypes()
df_work2authorships.rename(columns={"id": "work_id"}, inplace=True)

In [ ]:
df_work2authorships.head(3)

In [ ]:
# Expandir la lista de authorships
df_work2authorships_exploded = df_work2authorships.explode('authorships', ignore_index=True)

In [ ]:
df_work2authorships_exploded.head(3)

In [ ]:
# Normalizar la información de authorships
df_authorships_norm = pd.json_normalize(df_work2authorships_exploded['authorships'])
df_authorships_norm.rename(columns={"author.id": "author_id"}, inplace=True)

In [ ]:
df_authorships_norm.head(3)

In [ ]:
# Combinar work_id con la información normalizada de authorships
df_work2authorships = df_work2authorships_exploded[['work_id']].join(df_authorships_norm)

In [ ]:
df_work2authorships.head(3)

In [ ]:
# Extraer la relación work-author
df_work2author = df_work2authorships[['work_id', 'author_id', 'author_position']]

In [ ]:
df_work2author.head(3)

In [ ]:
# Expandir la lista de instituciones asociadas a cada autor
df_work2institution_exploded = df_work2authorships.explode('institutions', ignore_index=True)

In [ ]:
df_work2institution_exploded.head(3)

In [ ]:
# Normalizar la información de instituciones
df_institution_norm = pd.json_normalize(df_work2institution_exploded['institutions'])
df_institution_norm.drop(columns=['lineage'], errors='ignore', inplace=True)

In [ ]:
df_institution_norm.head(3)

In [ ]:
# Combinar author_id con la información normalizada de instituciones
df_author2institution = df_work2institution_exploded[['author_id']].join(df_institution_norm)

In [ ]:
df_author2institution.head(3)

In [ ]:
# Combinar work_id con la información normalizada de instituciones
df_work2institution = df_work2institution_exploded[['work_id']].join(df_institution_norm)

In [ ]:
df_work2institution.head(3)

# Nodo

In [ ]:
def openalex_load_work_authorships(df_work_raw):

    # Seleccionar las columnas necesarias y convertir los tipos de datos
    df_work2authorships = df_work_raw[['id', 'authorships']].convert_dtypes()
    df_work2authorships.rename(columns={"id": "work_id"}, inplace=True)

    # Expandir la lista de authorships
    df_work2authorships_exploded = df_work2authorships.explode('authorships', ignore_index=True)

    # Normalizar la información de authorships
    df_authorships_norm = pd.json_normalize(df_work2authorships_exploded['authorships'])
    df_authorships_norm.rename(columns={"author.id": "author_id"}, inplace=True)
    
    # Combinar work_id con la información normalizada de authorships
    df_work2authorships = df_work2authorships_exploded[['work_id']].join(df_authorships_norm)

    # Extraer la relación work-author
    df_work2author = df_work2authorships[['work_id', 'author_id', 'author_position']]

    # Expandir la lista de instituciones asociadas a cada autor
    df_work2institution_exploded = df_work2authorships.explode('institutions', ignore_index=True)

    # Normalizar la información de instituciones
    df_institution_norm = pd.json_normalize(df_work2institution_exploded['institutions'])
    df_institution_norm.drop(columns=['lineage'], errors='ignore', inplace=True)

    # Combinar author_id con la información normalizada de instituciones
    df_author2institution = df_work2institution_exploded[['author_id']].join(df_institution_norm)

    # Combinar work_id con la información normalizada de instituciones
    df_work2institution = df_work2institution_exploded[['work_id']].join(df_institution_norm)
    
    df_work2author['_load_datetime'] = datetime.today()
    df_work2institution['_load_datetime'] = datetime.today()
    df_author2institution['_load_datetime'] = datetime.today()

    return df_work2author, df_work2institution, df_author2institution


## Ejecuto Nodo

In [ ]:
df_work2author, df_work2institution, df_author2institution = openalex_load_work_authorships(df_work_raw)

# Resultados

In [ ]:
df_work2author

In [ ]:
df_work2institution

In [ ]:
df_author2institution